In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format('csv').option('inferSchema',True).option('header',True).load('/Volumes/workspace/first_schema/big_data/BigMart Sales.csv')

In [0]:
df.display()

In [0]:
df.printSchema()

In [0]:
df.filter(col('Item_Fat_Content')=="Low Fat").display()

In [0]:
df.filter(col('Item_Fat_Content')=="LF").display()

### Modifying existing values in column

In [0]:
df = df.withColumn('Item_Fat_Content',regexp_replace('Item_Fat_Content','LF','Low Fat'))

### SCENARIO Fetch records with item type as soft drink and item weight less than 10

In [0]:
df.filter((col('Item_Type')=='Soft Drinks') & (col('Item_Weight')<10)).display()

### Scenario - Fetch the data with tier in (Tier1 or Tier 2) and outlet size is null

In [0]:
df.filter((col('Outlet_Location_Type').isin('Tier 1','Tier 2')) & (col('Outlet_Size').isNull())).display()

### Split & Indexing

In [0]:
df_exp = df.select('Outlet_Type').withColumn('Outlet_Type',split('Outlet_Type',' '))

### Explode Function

In [0]:
df_exp.withColumn('Outlet_Type',explode('Outlet_Type')).display()

In [0]:
df.select('Item_Type').filter(col('Item_Type')=='Frozen Foods').count()

### Get Average MRP by Item Type

In [0]:
df.groupBy('Item_Type').agg(avg('Item_MRP').alias('avg_mrp'),sum('Item_MRP').alias('Total Sales'),count('Item_Type').alias('No of Items Sold')).display()

In [0]:
df.groupBy('Item_Type').pivot('Outlet_Type').agg(sum('Item_MRP')).display()

### Case When

In [0]:
df.withColumn('Veg flag',when(col('Item_Type')=='Meat','Non Veg').otherwise('Veg')).display()

### Window functions

In [0]:
from pyspark.sql.window import Window

In [0]:
# Rank Item_Identifier

df.withColumn('rnk',rank().over(Window.orderBy(col('Item_Identifier').desc()))).display()

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.partitionBy('Item_Type').orderBy(col('Item_Type')).rowsBetween(Window.unboundedPreceding,Window.currentRow))).display()

In [0]:
df.withColumn('cumsum',sum('Item_MRP').over(Window.partitionBy('Item_Type').orderBy(col('Item_Type')).rowsBetween(-2,0))).select('Item_Type','Item_MRP','cumsum').display()